In [1]:

# ==============================================================================
# Módulo de Extracción Profunda y Anti-Duplicados: Mitula (Arriendos)
# ==============================================================================

import os
import time
import re
import base64  
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# --- CONFIGURACIÓN PARA ASIGNACIÓN DE PÁGINAS---
PAGINA_INICIO = 1  
PAGINA_FIN = 1  
# ----------------------------------------------

os.environ["DISPLAY"] = ":99"  
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")
print(f"Entorno virtual listo. Asignación Mitula: Páginas {PAGINA_INICIO} a la {PAGINA_FIN}.")

options = Options()
# options.add_argument("--headless=new") # Comentado para que veas la magia
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)
    url_base = "https://casas.mitula.cl/find?operationType=rent&propertyType=apartment&geoId=R231672"

    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN EL CATÁLOGO (PRECIOS, IDs Y ENLACES)
    # ==============================================================================
    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):
        print(f"\n--- [FASE 1] Procesando Página {pagina_actual} ---")
        
        url_pagina = f"{url_base}&page={pagina_actual}"
        driver.get(url_pagina)

        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.listing-card"))
            )
            
            for _ in range(4):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)
            time.sleep(1) 

            bloques = driver.find_elements(By.CSS_SELECTOR, "article.listing-card")
            
            for bloque in bloques:
                try:
                    # 1. LA LLAVE MAESTRA: El ID único del departamento
                    listing_id = bloque.get_attribute("data-listingid")
                    if not listing_id: continue # Si no tiene ID, lo ignoramos por seguridad
                    
                    # 2. Filtro de Ubicación
                    texto_tarjeta = bloque.get_attribute("textContent").lower()
                    if "coquimbo" in texto_tarjeta: ubicacion = "Coquimbo"      
                    elif "la serena" in texto_tarjeta: ubicacion = "La Serena"     
                    else: continue 

                    # 3. Desencriptador de Enlaces Base64
                    url = "Sin URL"
                    try:
                        codigo_secreto = bloque.get_attribute("data-clickdestination")
                        if codigo_secreto:
                            url_decodificada = base64.b64decode(codigo_secreto).decode('utf-8', errors='ignore')
                            url = "https://casas.mitula.cl" + url_decodificada
                        else:
                            url = f"https://casas.mitula.cl/propiedad_id_{listing_id}"
                    except: pass

                    if url == "Sin URL" or "javascript" in url: continue

                    # 4. Extracción de Precio Base
                    try:
                        precio_texto = bloque.find_element(By.CSS_SELECTOR, ".price__actual").get_attribute("textContent")
                        v_limpio = precio_texto.replace("$", "").replace("/mes", "").replace(".", "").replace(",", "").strip()
                        precio = float(v_limpio) if v_limpio.isdigit() else 0.0
                    except: precio = 0.0
                        
                    if precio == 0.0: continue 

                    # 5. Extracción Numérica Base
                    m2, dormitorios, banos = 0, 0, 0
                    match_m2 = re.search(r'(\d+)\s*(m2|m²|mts)', texto_tarjeta)
                    if match_m2: m2 = int(match_m2.group(1))

                    match_dorm = re.search(r'(\d+)\s*dormitorio', texto_tarjeta)
                    if match_dorm: dormitorios = int(match_dorm.group(1))

                    match_bano = re.search(r'(\d+)\s*baño', texto_tarjeta)
                    if match_bano: banos = int(match_bano.group(1))

                    # Guardamos en la sala de espera
                    catalogo_urls.append({
                        "responsable": "Constanza Torres", # Aqui ponen su nombre
                        "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "listing_id": listing_id, # GUARDAMOS EL ID
                        "titulo": "Departamento Mit", 
                        "ubicacion": ubicacion,
                        "m2": m2,
                        "precio": precio,
                        "dormitorios": dormitorios,
                        "baños": banos,
                        "url": url
                    })

                except Exception as e:
                    continue 

            print(f"Se recolectaron enlaces base de la página {pagina_actual}.")

        except TimeoutException:
            print("Advertencia: Mit tardó en cargar. Posible fin de resultados.")
            continue
            
    print(f"\nCatálogo listo: {len(catalogo_urls)} propiedades encontradas. Iniciando Fase 2...")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA (Buscando amenidades ocultas)
    # ==============================================================================
    for i, propiedad in enumerate(catalogo_urls):
        if (i + 1) % 5 == 0:
            print(f"Progreso: Buscando en la propiedad {i + 1} de {len(catalogo_urls)}...")
        
        # Inicializamos en 0
        propiedad["estacionamiento"] = 0
        propiedad["piscina"] = 0
        propiedad["quincho"] = 0
        propiedad["terraza"] = 0
        propiedad["gimnasio"] = 0
        propiedad["lavanderia"] = 0

        try:
            driver.get(propiedad["url"])
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(1.5) 
            
            # Raspado Masivo del Body (Incluye descripción completa)
            texto_total = driver.find_element(By.TAG_NAME, "body").get_attribute("textContent").lower()
            
            if "estacionamiento" in texto_total: propiedad["estacionamiento"] = 1
            if "piscina" in texto_total: propiedad["piscina"] = 1
            if "quincho" in texto_total: propiedad["quincho"] = 1
            if "terraza" in texto_total: propiedad["terraza"] = 1
            if "gimnasio" in texto_total: propiedad["gimnasio"] = 1
            if "lavander" in texto_total: propiedad["lavanderia"] = 1

        except Exception as e:
            pass # Si falla, guarda la propiedad con amenidades en 0
            
        datos_finales.append(propiedad)

except Exception as e:
    print(f"Error crítico en Selenium: {e}")

finally:
    if driver is not None: driver.quit()

# ==============================================================================
# GUARDADO EN MONGODB (UPSERT CON LISTING_ID)
# ==============================================================================
print("\n--- Iniciando guardado en Base de Datos ---")
try:
    if datos_finales:
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Prueba_MIT"]
        coleccion = db["Scraping_MIT"] 

        registros_nuevos, registros_actualizados = 0, 0

        for d in datos_finales:
            # LA MAGIA ANTI-DUPLICADOS: Buscamos por listing_id, no por URL
            resultado = coleccion.update_one(
                {"listing_id": d["listing_id"]}, 
                {"$set": d}, 
                upsert=True
            )
            
            if resultado.upserted_id: registros_nuevos += 1
            else: registros_actualizados += 1

        print(f"¡Extracción Completada, {datos_finales[0]['responsable']}!")
        print(f"   -> {registros_nuevos} propiedades NUEVAS guardadas.")
        print(f"   -> {registros_actualizados} propiedades ACTUALIZADAS (sin duplicar).")
    else:
        print("No se extrajeron datos válidos.")
except Exception as e:
    print(f"Error de Base de Datos: {e}")

Entorno virtual listo. Asignación Mitula: Páginas 1 a la 1.

--- [FASE 1] Procesando Página 1 ---
Se recolectaron enlaces base de la página 1.

Catálogo listo: 28 propiedades encontradas. Iniciando Fase 2...
Progreso: Buscando en la propiedad 5 de 28...
Progreso: Buscando en la propiedad 10 de 28...
Progreso: Buscando en la propiedad 15 de 28...
Progreso: Buscando en la propiedad 20 de 28...
Progreso: Buscando en la propiedad 25 de 28...

--- Iniciando guardado en Base de Datos ---
¡Extracción Completada, Constanza Torres!
   -> 28 propiedades NUEVAS guardadas.
   -> 0 propiedades ACTUALIZADAS (sin duplicar).


In [6]:
from scraper_constanza_torres import ejecutar_extraccion


print(" Iniciando prueba scraping Mitula...\n")

datos = ejecutar_extraccion()

print(f"\n Total propiedades extraídas: {len(datos)}")

for i, d in enumerate(datos[:5]):
    print(f"\n Propiedad {i+1}")
    print(d)

ModuleNotFoundError: No module named 'scraper_constanza_torres'